# FinTracker — Analisi Finanziaria del Portafoglio

Questo notebook analizza le performance dei titoli presenti nella watchlist di FinTracker.  
Include analisi dei rendimenti, del rischio, delle correlazioni e un backtest della strategia RSI.

**Autore:** Scansani Elia  
**Data:** 2026

In [2]:
# Importazione librerie
import sys
import os

# Aggiunge la root del progetto al path
sys.path.append(os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# Stile dei grafici
plt.style.use("dark_background")
sns.set_palette("husl")

# Impostazioni display
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)

print("✅ Librerie importate correttamente")
print(f"📅 Data analisi: {datetime.now().strftime('%d/%m/%Y')}")

✅ Librerie importate correttamente
📅 Data analisi: 06/05/2026


In [11]:
# Connessione al database PostgreSQL per leggere i ticker in watchlist
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv

cartella_progetto = os.path.abspath(os.path.join(os.getcwd(), ".."))
env_path = os.path.join(cartella_progetto, ".env")
load_dotenv(env_path)

print(f"📁 Cartella progetto: {cartella_progetto}")
print(f"📄 File .env: {env_path}")
print(f"🔑 DB_HOST: {os.getenv('DB_HOST')}")
print(f"🔑 DB_NAME: {os.getenv('DB_NAME')}")
print(f"🔑 DB_USER: {os.getenv('DB_USER')}")
print(f"🔑 Password presente: {'✅' if os.getenv('DB_PASSWORD') else '❌'}")

def get_watchlist_db():
    """Legge i ticker dalla watchlist di FinTracker."""
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432"),
        dbname=os.getenv("DB_NAME", "fintracker"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD")
    )
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute("SELECT ticker, nome FROM watchlist ORDER BY data_aggiunta")
    righe = [dict(r) for r in cursor.fetchall()]
    conn.close()
    return righe

watchlist = get_watchlist_db()
tickers   = [r["ticker"] for r in watchlist]
nomi      = {r["ticker"]: r["nome"] for r in watchlist}

print(f"📋 Ticker in watchlist: {len(tickers)}")
for t in watchlist:
    print(f"   {t['ticker']:<10} {t['nome']}")

📁 Cartella progetto: C:\Users\scans\Desktop\tempo_libero
📄 File .env: C:\Users\scans\Desktop\tempo_libero\.env
🔑 DB_HOST: None
🔑 DB_NAME: None
🔑 DB_USER: None
🔑 Password presente: ❌


OperationalError: connection to server at "localhost" (::1), port 5432 failed: fe_sendauth: no password supplied


In [5]:
# Scarica 2 anni di dati storici per ogni ticker
PERIODO = "2y"

print(f"📡 Scaricando {PERIODO} di dati storici...")
print("-" * 40)

dati = {}
for ticker in tickers:
    try:
        df = yf.Ticker(ticker).history(period=PERIODO, interval="1d")
        if not df.empty:
            dati[ticker] = df
            print(f"  ✅ {ticker:<10} {len(df)} giorni di dati")
        else:
            print(f"  ⚠️  {ticker:<10} nessun dato disponibile")
    except Exception as e:
        print(f"  ❌ {ticker:<10} errore: {e}")

print("-" * 40)
print(f"✅ Dati scaricati per {len(dati)}/{len(tickers)} ticker")

📡 Scaricando 2y di dati storici...
----------------------------------------


NameError: name 'tickers' is not defined

In [6]:
# Costruisce un DataFrame unico con i prezzi di chiusura
# di tutti i ticker — una colonna per ticker, una riga per giorno

prezzi = pd.DataFrame({
    ticker: df["Close"]
    for ticker, df in dati.items()
})

# Rimuove giorni in cui tutti i mercati erano chiusi
prezzi = prezzi.dropna(how="all")

print(f"📊 Tabella prezzi: {prezzi.shape[0]} giorni × {prezzi.shape[1]} ticker")
print(f"   Dal: {prezzi.index[0].strftime('%d/%m/%Y')}")
print(f"   Al:  {prezzi.index[-1].strftime('%d/%m/%Y')}")
print()
print(prezzi.tail())

📊 Tabella prezzi: 0 giorni × 0 ticker


IndexError: index 0 is out of bounds for axis 0 with size 0